[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

### Steps:

Suppose:

- $Q, K, V \in \mathbb{R}^{B \times N \times D}$
- $B_r$ = number of query rows in the current Q block
- $B_c$ = number of key/value rows in the current K/V block
- $D$ = head dimension

For one query block:

$$
Q_i \in \mathbb{R}^{B_r \times D}
$$

we iterate over all key/value blocks:

$$
K_j, V_j \in \mathbb{R}^{B_c \times D}
$$

The goal is to compute the same result as normal attention:

$$
O =
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{D}}
\right)V
$$

without ever **materializing** the full $N \times N$ attention matrix.

---

# Step 1 — Initialize the online-softmax state

For every query row in the current Q block, we maintain three running quantities:

## Running maximum: $m$

$$
m^{(0)} = -\infty
$$

$m$ stores the largest attention score seen so far for each query row.

Shape:

$$
m \in \mathbb{R}^{B_r \times 1}
$$

```python
m = torch.full((B, Br, 1), float("-inf"), device=Q.device)
```

---

## Running denominator: $\ell$

$$
\ell^{(0)} = 0
$$

$\ell$ stores the running softmax denominator.

Shape:

$$
\ell \in \mathbb{R}^{B_r \times 1}
$$

```python
l = torch.zeros((B, Br, 1), device=Q.device)
```

---

## Running weighted-value accumulator: $A$

$$
A^{(0)} = 0
$$

$A$ stores the running **unnormalized weighted sum of V**.

Shape:

$$
A \in \mathbb{R}^{B_r \times D}
$$

```python
acc = torch.zeros((B, Br, D), device=Q.device)
```

---

# Step 2 — Compute the attention-score tile

For the current Q block $Q_i$ and K block $K_j$:

$$
S_{ij}
=
\frac{Q_iK_j^T}{\sqrt{D}}
$$

Shapes:

$$
Q_i \in \mathbb{R}^{B_r \times D}
$$

$$
K_j \in \mathbb{R}^{B_c \times D}
$$

therefore:

$$
S_{ij} \in \mathbb{R}^{B_r \times B_c}
$$

Each row corresponds to one query.

Each column corresponds to one key in the current K block.

```python
attn_scores = q @ k.transpose(-2, -1) / math.sqrt(D)
```

---

# Step 3 — Find the row maximum of the current score tile

For each query row, find the largest score in the current K block:

$$
m_{\text{block},i}
=
\max_{k \in \text{current K block}} S_{ik}
$$

Therefore:

$$
m_{\text{block}}
\in
\mathbb{R}^{B_r \times 1}
$$

```python
block_max = attn_scores.max(dim=-1, keepdim=True).values
```

For example:

$$
S =
\begin{bmatrix}
1 & 4 & 2 & 3 \\
5 & 1 & 2 & 0
\end{bmatrix}
$$

then:

$$
m_{\text{block}}
=
\begin{bmatrix}
4 \\
5
\end{bmatrix}
$$

---

# Step 4 — Update the running maximum

$m$ contains the largest score from all **previous K blocks**.

$m_{\text{block}}$ contains the largest score from the **current K block**.

So:

$$
m_{\text{new}}
=
\max
\left(
m_{\text{old}},
m_{\text{block}}
\right)
$$

```python
new_m = torch.maximum(m, block_max)
```

This means that for every query:

$$
m_{\text{new},i}
=
\max
\left(
\text{all scores seen so far for query } i
\right)
$$

After all K blocks have been processed:

$$
m_i
=
\max_{j=1,\ldots,N} S_{ij}
$$

which is exactly the row maximum used by numerically stable softmax.

---

# Step 5 — Rescale contributions from previous K/V blocks

The previous $\ell$ and $A$ were calculated relative to:

$$
m_{\text{old}}
$$

but now our stable-softmax reference point is:

$$
m_{\text{new}}
$$

Therefore we need the correction factor:

$$
\alpha
=
e^{m_{\text{old}} - m_{\text{new}}}
$$

```python
old_scale = torch.exp(m - new_m)
```

If the maximum did not change:

$$
m_{\text{old}} = m_{\text{new}}
$$

then:

$$
\alpha
=
e^0
=
1
$$

so the old values do not need to change.

If a larger maximum was discovered, then:

$$
m_{\text{new}} > m_{\text{old}}
$$

and therefore:

$$
0 < \alpha < 1
$$

which rescales the previously accumulated values.

---

# Step 6 — Compute unnormalized softmax weights for the current tile

For every score in the current score tile:

$$
P_{ij}
=
e^{S_{ij} - m_{\text{new},i}}
$$

```python
p = torch.exp(attn_scores - new_m)
```

Here `new_m` has shape:

$$
[B, B_r, 1]
$$

while `attn_scores` has shape:

$$
[B, B_r, B_c]
$$

so `new_m` is broadcast across all keys in each row.

Important:

$P$ is **not yet softmax**.

It contains the unnormalized softmax weights:

$$
P_{ij}
=
e^{S_{ij}-m_i}
$$

Actual softmax would be:

$$
\operatorname{softmax}(S_i)_j
=
\frac{
e^{S_{ij}-m_i}
}{
\sum_k e^{S_{ik}-m_i}
}
$$

So $P$ is the numerator before dividing by the denominator.

---

# Step 7 — Update the running softmax denominator

For the current K block, the denominator contribution is:

$$
\sum_{k \in \text{current block}}
e^{S_{ik}-m_{\text{new},i}}
$$

Since:

$$
P_{ik}
=
e^{S_{ik}-m_{\text{new},i}}
$$

this is simply:

$$
\sum_k P_{ik}
$$

In code:

```python
p.sum(dim=-1, keepdim=True)
```

But the denominator from previous blocks was calculated relative to $m_{\text{old}}$.

Therefore we first rescale it:

$$
\ell_{\text{old}}
e^{m_{\text{old}}-m_{\text{new}}}
$$

and then add the contribution from the current block:

$$
\boxed{
\ell_{\text{new}}
=
\ell_{\text{old}}
e^{m_{\text{old}}-m_{\text{new}}}
+
\sum_k P_{ik}
}
$$

```python
l = l * old_scale + p.sum(dim=-1, keepdim=True)
```

After all K blocks have been processed:

$$
\ell_i
=
\sum_{j=1}^{N}
e^{S_{ij}-m_i}
$$

This is exactly the denominator of numerically stable softmax.

---

# Step 8 — Update the running weighted-V accumulator

Normal attention output can be written as:

$$
O_i
=
\frac{
\sum_j e^{S_{ij}-m_i}V_j
}{
\sum_j e^{S_{ij}-m_i}
}
$$

The numerator is:

$$
\sum_j e^{S_{ij}-m_i}V_j
$$

We store this numerator incrementally in `acc`.

For the current K/V block:

$$
P_{ij}
=
e^{S_{ij}-m_{\text{new},i}}
$$

so its weighted-V contribution is:

$$
\sum_{j \in \text{current block}}
P_{ij}V_j
$$

In matrix form:

$$
PV
$$

```python
p @ v
```

But the previous accumulator was based on the old maximum.

Therefore it must first be rescaled:

$$
A_{\text{old}}
e^{m_{\text{old}}-m_{\text{new}}}
$$

Then add the current block contribution:

$$
\boxed{
A_{\text{new}}
=
A_{\text{old}}
e^{m_{\text{old}}-m_{\text{new}}}
+
PV
}
$$

```python
acc = acc * old_scale + p @ v
```

After every K/V block has been processed:

$$
A_i
=
\sum_{j=1}^{N}
e^{S_{ij}-m_i}V_j
$$

So:

- $\ell$ stores the softmax denominator
- $A$ stores the weighted-V numerator

---

# Step 9 — Save the new running maximum

The current maximum now becomes the old maximum for the next K/V block:

$$
m_{\text{old}}
\leftarrow
m_{\text{new}}
$$

```python
m = new_m
```

Then we move to the next K/V tile and repeat Steps 2–9.

---

# Step 10 — Normalize after ALL K/V blocks

Once every K/V block has been processed, we have:

$$
A_i
=
\sum_{j=1}^{N}
e^{S_{ij}-m_i}V_j
$$

and:

$$
\ell_i
=
\sum_{j=1}^{N}
e^{S_{ij}-m_i}
$$

Therefore:

$$
\boxed{
O_i
=
\frac{A_i}{\ell_i}
}
$$

Expanding:

$$
O_i
=
\frac{
\sum_j e^{S_{ij}-m_i}V_j
}{
\sum_j e^{S_{ij}-m_i}
}
$$

But:

$$
\frac{
e^{S_{ij}-m_i}
}{
\sum_k e^{S_{ik}-m_i}
}
=
\operatorname{softmax}(S_i)_j
$$

therefore:

$$
O_i
=
\sum_j
\operatorname{softmax}(S_i)_jV_j
$$

which is exactly normal attention.

```python
out[:, qs:qs + Br] = acc / l
```

---

# The Complete Online-Softmax Recurrence

For every Q tile and K/V tile:

## 1. Attention scores

$$
S
=
\frac{Q_iK_j^T}{\sqrt D}
$$

## 2. Maximum of current tile

$$
m_{\text{block}}
=
\max_{\text{row}}(S)
$$

## 3. Update running maximum

$$
m_{\text{new}}
=
\max
\left(
m_{\text{old}},
m_{\text{block}}
\right)
$$

## 4. Compute correction for old state

$$
\alpha
=
e^{m_{\text{old}}-m_{\text{new}}}
$$

## 5. Compute current unnormalized softmax weights

$$
P
=
e^{S-m_{\text{new}}}
$$

## 6. Update softmax denominator

$$
\boxed{
\ell_{\text{new}}
=
\alpha\ell_{\text{old}}
+
\sum_{\text{row}}P
}
$$

## 7. Update weighted-V accumulator

$$
\boxed{
A_{\text{new}}
=
\alpha A_{\text{old}}
+
PV
}
$$

## 8. Update maximum

$$
m_{\text{old}}
\leftarrow
m_{\text{new}}
$$

After all K/V blocks:

$$
\boxed{
O
=
\frac{A}{\ell}
}
$$

---

# Why this equals normal attention

Normal attention computes:

$$
S
=
\frac{QK^T}{\sqrt D}
$$

then:

$$
P
=
\operatorname{softmax}(S)
$$

then:

$$
O
=
PV
$$

For one query row:

$$
O_i
=
\frac{
\sum_j e^{S_{ij}-m_i}V_j
}{
\sum_j e^{S_{ij}-m_i}
}
$$

FlashAttention computes the exact same numerator and denominator, but accumulates them **one K/V tile at a time**.

Instead of creating:

$$
S \in \mathbb{R}^{N\times N}
$$

we only create:

$$
S_{ij}
\in
\mathbb{R}^{B_r\times B_c}
$$

for one tile at a time.

---

# Shape Summary

For a single-head implementation:

$$
Q,K,V
:
[B,N,D]
$$

Current Q tile:

$$
q:
[B,B_r,D]
$$

Current K/V tiles:

$$
k,v:
[B,B_c,D]
$$

Attention-score tile:

$$
S:
[B,B_r,B_c]
$$

Running maximum:

$$
m:
[B,B_r,1]
$$

Running denominator:

$$
\ell:
[B,B_r,1]
$$

Running weighted-V accumulator:

$$
A:
[B,B_r,D]
$$

Current unnormalized softmax weights:

$$
P:
[B,B_r,B_c]
$$

Final output block:

$$
O_i:
[B,B_r,D]
$$

---

# Mental Model

For each fixed Q block:

$$
Q_i
$$

scan through:

$$
(K_1,V_1),
(K_2,V_2),
\ldots,
(K_T,V_T)
$$

while carrying only:

$$
m,\ell,A
$$

from one K/V block to the next.

Conceptually:

$$
(K_1,V_1)
\rightarrow
(m_1,\ell_1,A_1)
$$

$$
(K_2,V_2)
\rightarrow
(m_2,\ell_2,A_2)
$$

$$
(K_3,V_3)
\rightarrow
(m_3,\ell_3,A_3)
$$

$$
\cdots
$$

Finally:

$$
O_i
=
\frac{A_T}{\ell_T}
$$

The full $N\times N$ attention-score matrix is never required.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [3]:
from torch_judge import hint
hint("flash_attention")


💡 Hint for Flash Attention (Tiled):
   Process Q in blocks. For each Q-block, iterate over K/V blocks. Use online softmax: track running max and sum, rescale accumulator when max changes. output = acc / row_sum.



In [24]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    # pass
    
    B, N, D = Q.shape
    out = torch.empty_like(Q)

    for qs in range(0, N, block_size):
        q = Q[:, qs:qs + block_size]
        Br = q.shape[1] # last block might be smaller than block_size
        
        m = torch.full((B, Br, 1), float('-inf'), device=Q.device) # running max for numerical stability

        l = torch.zeros((B, Br, 1), device=Q.device) # running softmax denominator
        
        acc = torch.zeros((B, Br, D), device=Q.device) # running weighted sum of the V vectors
        
        for ks in range(0, N, block_size):
            k = K[:, ks:ks + block_size]
            v = V[:, ks:ks + block_size]

            # Compute attention scores
            attn_scores = q @ k.transpose(-2, -1) / math.sqrt(D)  # (B, block_size, block_size)

            block_max = attn_scores.max(dim=-1, keepdim=True).values 
            new_m = torch.maximum(m, block_max)  # update per-query running max across all K/V blocks seen so far
            
            # rescale old contribution because max changed
            old_scale = torch.exp(m - new_m)

            # probabilities relative to new running max
            p = torch.exp(attn_scores - new_m)
            
            # update denominator(row sum of exp)
            l = l * old_scale + p.sum(dim=-1, keepdim=True)
            
            # update weighted V accumulator
            acc = acc * old_scale + p @ v

            # update max
            m = new_m

        # normalize only after ALL K/V blocks
        out[:, qs:qs + Br] = acc / l
    
    return out
        

In [22]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

q: torch.Size([1, 4, 4])
m: torch.Size([1, 4, 1])
l: torch.Size([1, 4, 1])
acc: torch.Size([1, 4, 4])
q: torch.Size([1, 4, 4])
m: torch.Size([1, 4, 1])
l: torch.Size([1, 4, 1])
acc: torch.Size([1, 4, 4])
q: torch.Size([1, 4, 4])
m: torch.Size([1, 4, 1])
l: torch.Size([1, 4, 1])
acc: torch.Size([1, 4, 4])
q: torch.Size([1, 4, 4])
m: torch.Size([1, 4, 1])
l: torch.Size([1, 4, 1])
acc: torch.Size([1, 4, 4])
Match: True


In [23]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
q: torch.Size([2, 4, 8])
m: torch.Size([2, 4, 1])
l: torch.Size([2, 4, 1])
acc: torch.Size([2, 4, 8])
q: torch.Size([2, 4, 8])
m: torch.Size([2, 4, 1])
l: torch.Size([2, 4, 1])
acc: torch.Size([2, 4, 8])
q: torch.Size([2, 4, 8])
m: torch.Size([2, 4, 1])
l: torch.Size([2, 4, 1])
acc: torch.Size([2, 4, 8])
q: torch.Size([2, 4, 8])
m: torch.Size([2, 4, 1])
l: torch.Size([2, 4, 1])
acc: torch.Size([2, 4, 8])
q: torch.Size([2, 4, 8])
m: torch.Size([2, 4, 1])
l: torch.Size([2, 4, 1])
acc: torch.Size([2, 4, 8])
q: torch.Size([2, 4, 8])
m: torch.Size([2, 4, 1])
l: torch.Size([2, 4, 1])
acc: torch.Size([2, 4, 8])
q: torch.Size([2, 4, 8])
m: torch.Size([2, 4, 1])
l: torch.Size([2, 4, 1])
acc: torch.Size([2, 4, 8])
q: torch.Size([2, 4, 8])
m: torch.Size([2, 4, 1])
l: torch.Size([2, 4, 1])
acc: torch.Size([2, 4, 8])
q: torch.Size([2, 4, 8])
m: torch.Size([2, 4, 1])
l: torch.Size([2, 4, 1])
acc: torch.Size